# Geospatial visualisation with Folium

Location is a first-class feature in many datasets. This notebook maps USGS earthquakes as markers,
draws a choropleth of life expectancy by country, and computes proximity — which events are nearest
a chosen point. It uses the shared `charts.py` builders.

## Learning objectives

By the end of this notebook you will be able to:

- place point markers on an interactive map with Folium;
- build a choropleth by joining values to country names;
- compute great-circle distance to find events near a location;
- choose a basemap and explain the trade-offs between map types;
- recognise the pitfalls of plotting geographic data.

## Concept

A **marker map** puts one symbol per observation at its coordinates. It answers "where did this
happen?" and is at its best when points are sparse enough to distinguish. A **choropleth** shades
regions by a value: it answers "how does this measure vary by area?" and requires a join between
the data and the region boundaries, usually by name or code. Folium builds Leaflet maps, and the
tiles come from a basemap provider, which means a displayed map needs network access even though
creating the object does not.

**Proximity** is a distance question. The haversine formula gives the great-circle distance
between two latitude/longitude points on a sphere; it is accurate to a fraction of a percent,
which is plenty for a map. Sorting events by distance is the basis of "nearest earthquake to me".

Geographic pitfalls are everywhere: a choropleth hides variation within a region, raw counts make
large countries look important while rates do not, and a projection distorts area and shape.
Always state what the colour and size actually encode.

## Worked example

### Load data and build a marker map

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import charts
from ds_practice import load_gapminder, load_usgs_quakes, set_seed

set_seed(42)
quakes = load_usgs_quakes()
quakes = quakes.dropna(subset=["latitude", "longitude", "mag"])
print("earthquakes:", len(quakes), "| magnitude range:", quakes["mag"].min(), "-", quakes["mag"].max())
display(quakes[["place", "mag", "latitude", "longitude"]].head())

In [ ]:
strongest = quakes.nlargest(50, "mag")
earthquake_map = charts.folium_map(strongest, lat="latitude", lon="longitude",
                                   popup="place", zoom_start=2)
earthquake_map

### A choropleth by country

Gapminder country names align with the built-in country-name locations, so we can shade each
country by life expectancy in the latest year.

In [ ]:
gap = load_gapminder()
latest = gap[gap["year"] == gap["year"].max()]
choropleth = charts.choropleth(
    latest, locations="country", values="lifeExp",
    title="Life expectancy by country, 2007", locationmode="country names",
    color_continuous_scale="Viridis",
)
choropleth.show()

### Proximity

The haversine formula returns kilometres between two points. We measure every strong earthquake
from Reykjavík and list the closets ten.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))

reykjavik = (64.1466, -21.9426)
near = quakes.assign(distance_km=haversine_km(
    reykjavik[0], reykjavik[1], quakes["latitude"].to_numpy(), quakes["longitude"].to_numpy()
))
nearest = near.nsmallest(10, "distance_km")[["place", "mag", "distance_km"]]
display(nearest.round(1))

### Focus the map on a region

Passing a filtered frame re-centres the map. The function always centres on the mean coordinates,
so we hand it only the events of interest.

In [ ]:
nordic = near[near["distance_km"] < 2000]
charts.folium_map(nordic, lat="latitude", lon="longitude", popup="place", zoom_start=4)

## Exercises

1. **Magnitude circles.** Modify the map so marker radius grows with magnitude. Explain why a
   linear radius mapping overstates differences and area is the fairer encoding.
2. **Counts versus rates.** Build two choropleth-style summaries of earthquakes per country using
   the `place` text, and argue why a count favours large countries.
3. **Proximity from home.** Change the reference point to a city you know, list the five nearest
   earthquakes, and report the distances. Note any dependence on the 30-day feed window.

## Limitations

The USGS feed covers only the last 30 days, so it is a moving snapshot rather than a long record.
Folium maps need a network connection to fetch basemap tiles when displayed, which is why this
notebook builds map objects but should be viewed online. Country-name joins fail silently when a
name does not match the built-in list, dropping rows from a choropleth. Finally, the haversine
formula assumes a spherical Earth, and `place` strings are not a reliable country field.